# 🛠️ Data-stack-dev-env: The Ultimate Spark, SparkSQL & ClearML Model Governance Playbook

Welcome to the developer playground and production reference guide for your machine learning and data engineering stack. This notebook acts as an **all-inclusive, syntax-focused cheatsheet** designed to provide direct reference implementations of Spark Core, SparkSQL, Spark MLlib, JDBC database connections, and ClearML model governance API calls.

---

## 1. Imports, Initialization & Configurations

This section outlines all required import statements across the entire Spark ecosystem (Core, SQL, and MLlib classification, regression, clustering, neural networks, and evaluation metrics), as well as configuration environment variables.

In [ ]:
import os
import sys
import pandas as pd

# ── Spark Core & SQL Setup ──
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ── Spark MLlib Feature Engineering & Pipeline ──
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler,
    MinMaxScaler,
    StringIndexer,
    OneHotEncoder,
    VectorIndexer,
    PCA
)

# ── Spark MLlib Supervised Learning (Regression) ──
from pyspark.ml.regression import (
    LinearRegression,
    GeneralizedLinearRegression,
    DecisionTreeRegressor,
    RandomForestRegressor,
    GBTRegressor
)

# ── Spark MLlib Supervised Learning (Classification) ──
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
    LinearSVC,
    NaiveBayes
)

# ── Spark MLlib Unsupervised Learning (Clustering) ──
from pyspark.ml.clustering import (
    KMeans,
    BisectingKMeans,
    GaussianMixture,
    LDA
)

# ── Spark MLlib Neural Networks ──
from pyspark.ml.classification import MultilayerPerceptronClassifier

# ── Spark MLlib Evaluators ──
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)

# ── ClearML SDK ──
from clearml import Task, OutputModel, InputModel, Model, Dataset

print("All libraries successfully imported.")

## 2. Start PySpark Session

Initializes the local `SparkSession` configured with resources, executors, dynamic allocations, and custom jars (e.g., PostgreSQL JDBC driver).

In [ ]:
# Configuration values mapped directly to the local dev environment stack
APP_NAME = "DataStackPlaybook-Spark"
MASTER_URL = "local[*]"  # Local Spark execution on all available cores
JDBC_JAR_PATH = os.environ.get('SPARK_JDBC_JAR', '/home/jovyan/jars/postgresql-42.7.3.jar') # Mapped via docker-compose

# Initialize SparkSession with Postgres JDBC driver configurations
spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master(MASTER_URL)
    .config("spark.jars", JDBC_JAR_PATH)
    .config("spark.sql.shuffle.partitions", "8") # optimized partition size for single host testing
    .getOrCreate()
)

# Adjust logging verbosity (options: ALL, DEBUG, INFO, WARN, ERROR, FATAL, OFF)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} Session initialized.")

## 3. PostgreSQL Database Connections via JDBC

Establish database connections to read and write database tables using Apache Spark's JDBC connectors with real stack variables.

In [ ]:
# PostgreSQL stack configurations automatically sourced from environment variables
PG_HOST     = os.environ.get('POSTGRES_HOST',     'postgres')
PG_PORT     = os.environ.get('POSTGRES_PORT',     '5432')
PG_DB       = os.environ.get('POSTGRES_DB',       'postgres')
PG_USER     = os.environ.get('POSTGRES_USER',     'postgres')
PG_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'postgres')

# JDBC connection string and properties
JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"
JDBC_PROPS = {
    "user": PG_USER,
    "password": PG_PASSWORD,
    "driver": "org.postgresql.Driver"
}

# ── A. WRITING DATA TO POSTGRESQL ──
# Setup dummy data to demonstrate and test RDBMS writes
write_test_data = [("user_alpha", 150), ("user_beta", 220)]
write_test_schema = T.StructType([
    T.StructField("username", T.StringType(), True),
    T.StructField("score", T.IntegerType(), True)
])
test_df = spark.createDataFrame(write_test_data, write_test_schema)

# Write data to PostgreSQL table via JDBC
# Save modes: 'overwrite', 'append', 'ignore', 'errorifexists'
test_df.write.jdbc(
    url=JDBC_URL,
    table="jdbc_connection_test",
    mode="overwrite",
    properties=JDBC_PROPS
)
print("Write test data to PostgreSQL completed.")

# ── B. READING DATA FROM POSTGRESQL ──
# Read the table back into a Spark DataFrame
loaded_test_df = spark.read.jdbc(
    url=JDBC_URL,
    table="jdbc_connection_test",
    properties=JDBC_PROPS
)
print("Read test data back from PostgreSQL:")
loaded_test_df.show()

## 4. ClearML MLOps & Model Governance

Establishing connections, tracking parameters, logging models manually or automatically, and versioning datasets using the ClearML API.

In [ ]:
# ── A. TASK INITIALIZATION & CONFIGURATION ──
# Task init creates a traceable experiment on your local ClearML server
task = Task.init(
    project_name="DataStackPlaybook",
    task_name="cheatsheet_syntax_guide",
    task_type=Task.TaskTypes.training,
    reuse_last_task_id=False
)

# Connect tracking configurations/hyperparameters to ClearML
run_config = {
    "postgres_db": PG_DB,
    "jdbc_driver": "org.postgresql.Driver",
    "spark_master": "local[*]"
}
task.connect(run_config)
print(f"ClearML Task initialized with ID: {task.id}")


# ── B. MANUAL MODEL REGISTRATION (`OutputModel`) ──
# Use OutputModel to manually register a model configuration and upload files
output_model = OutputModel(task=task, name="cheatsheet_output_model", framework="PySpark")

# Note: Call update_weights with a local path to upload model assets
# output_model.update_weights(weights_filename="path/to/model")

# Register label classes map
output_model.update_labels({"label_0": 0, "label_1": 1})

# Record design performance/evaluation metrics to registry
output_model.update_design_metrics({"r2": 0.85, "accuracy": 0.90})

# Set custom metadata keys
output_model.set_metadata({"environment": "staging", "data_version": "v1.0"})

# Publish model (Locks model registry metadata, transitioning status to Published)
output_model.publish()
print(f"OutputModel registered & published in ClearML Registry. ID: {output_model.id}")


# ── C. REUSING MODELS & TRACKING LINEAGE (`InputModel`) ──
# Load the model back in another task to preserve lineage links
input_model = InputModel(model_id=output_model.id)
task.connect(input_model) # logs input-model dependency under experiment configuration

# Download model weights to local cache directory
local_path = input_model.get_weights()
print(f"Input model weights cached locally at: {local_path}")


# ── D. MODEL REGISTRY QUERIES ──
# Search for models programmatically
models = Model.query_models(
    project_name="DataStackPlaybook",
    tags=["v1.0"],
    only_published=True
)
print(f"Found {len(models)} model(s) matching criteria.")


# ── E. DATASET GOVERNANCE (`Dataset`) ──
# Create versioned and tracked dataset storage in ClearML registry
dataset = Dataset.create(dataset_name="demo_cheatsheet_dataset", dataset_project="DataStackPlaybook")

# Add files or folders containing data split
# dataset.add_files("path/to/data_files")

# Upload and finalize (finalizing locks the dataset as read-only)
# dataset.upload()
# dataset.finalize()
print(f"ClearML Dataset created with ID: {dataset.id}")

## 5. Spark Data Engineering & ETL Operations

Detailed guide containing syntax for common row-level and column-level Spark DataFrame API operations.

In [ ]:
# Setup synthetic CSV file for DataFrame ETL demonstration
demo_csv_data = "id,name,amount,category\n1,alpha,100.5,A\n2,beta,200.0,B\n3,gamma,NaN,A\n3,gamma,NaN,A\n4,delta,150.2,C\n"
with open("demo_data.csv", "w") as f:
    f.write(demo_csv_data)

# ── A. READING DATA ──
# Reading CSV files with options
df_raw = spark.read.options(header="true", inferSchema="true", delimiter=",").csv("demo_data.csv")

# ── B. EXPLORATORY OPERATIONS ──
row_count = df_raw.count()
df_raw.printSchema()
df_raw.show()

# ── C. CLEANING & TRANSFORMATIONS ──
# Drop duplicates (distinct rows)
df_no_dupes = df_raw.dropDuplicates()

# Drop missing / null values
df_clean = df_no_dupes.dropna(subset=["amount"])

# Rename column
df_renamed = df_clean.withColumnRenamed("category", "category_group")

# Type Casting & Adding/Modifying Columns
df_cast = df_renamed.withColumn("amount_double", F.col("amount").cast(T.DoubleType()))

# Selecting specific columns
df_select = df_cast.select("id", "name", "amount_double", "category_group")

# Filtering rows
df_filtered = df_select.filter(F.col("amount_double") >= 100.0)

# Grouping and aggregations
df_grouped = df_filtered.groupBy("category_group").agg(
    F.count("id").alias("count_records"),
    F.sum("amount_double").alias("sum_amount"),
    F.avg("amount_double").alias("avg_amount")
)
print("Processed ETL results:")
df_grouped.show()

# ── D. WRITING DATA ──
# Save DataFrame as Parquet
df_filtered.write.mode("overwrite").parquet("etl_output.parquet")
print("Data written as Parquet successfully.")

## 6. In-Memory Manipulation using SparkSQL

Structured SQL queries executed inside Spark memory by registering views.

In [ ]:
# Register a DataFrame as a temporary in-memory SQL view
df_filtered.createOrReplaceTempView("v_etl_filtered")

# Run SQL statement directly
sql_df = spark.sql("""
    SELECT 
        id,
        name,
        amount_double,
        CASE 
            WHEN amount_double > 150.0 THEN 'VIP'
            ELSE 'Standard'
        END as user_tier
    FROM v_etl_filtered
    WHERE name IS NOT NULL
    ORDER BY amount_double DESC
""")

# Show SQL output
print("SparkSQL query output:")
sql_df.show()

## 7. PySpark Machine Learning (MLlib) Pipeline (Genuine Syntax)

All code cells in this section are runnable, syntax-focused implementations. A generic mock DataFrame is initialized first to supply inputs for the transformers, estimators, training, evaluation, and serialization steps.

In [ ]:
# ── A. INITIALIZE DUMMY DATAFRAME FOR ML SYNTAX COMPILATION ──
# We create a generic mock DataFrame to ensure all downstream ML syntax cells are runnable
mock_data = [
    (1.0, 2.0, 3.0, "A", 10.0, 1.0),
    (2.0, 1.5, 4.2, "B", 15.0, 0.0),
    (3.5, 3.0, 2.8, "A", 8.0, 1.0),
    (4.0, 2.2, 5.1, "C", 20.0, 0.0),
]
mock_schema = T.StructType([
    T.StructField("col1", T.DoubleType(), True),
    T.StructField("col2", T.DoubleType(), True),
    T.StructField("col3", T.DoubleType(), True),
    T.StructField("categorical_string_col", T.StringType(), True),
    T.StructField("target_label", T.DoubleType(), True),
    T.StructField("binary_label", T.DoubleType(), True),
])
df_ml = spark.createDataFrame(mock_data, mock_schema)
print("Mock DataFrame 'df_ml' created.")

### A. Feature Engineering & Preprocessing Transformers

In [ ]:
# 1. VectorAssembler: Merges columns into a single vector column
assembler = VectorAssembler(
    inputCols=["col1", "col2", "col3"],
    outputCol="features_vector"
)

# 2. StandardScaler: Scales features vector to unit variance
scaler = StandardScaler(
    inputCol="features_vector",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

# 3. MinMaxScaler: Rescales feature vectors to range [min, max]
min_max_scaler = MinMaxScaler(
    inputCol="features_vector",
    outputCol="scaled_features_minmax",
    min=0.0,
    max=1.0
)

# 4. StringIndexer: Maps categorical strings to numerical indices
string_indexer = StringIndexer(
    inputCol="categorical_string_col",
    outputCol="category_indexed",
    handleInvalid="keep"
)

# 5. OneHotEncoder: Maps numerical indices to binary vectors
ohe = OneHotEncoder(
    inputCols=["category_indexed"],
    outputCols=["category_ohe_vectors"]
)

# 6. VectorIndexer: Automatically indexes categorical columns in a vector
vector_indexer = VectorIndexer(
    inputCol="features_vector",
    outputCol="indexed_features_vector",
    maxCategories=5
)

### B. Supervised Learning (Regression Estimators)

In [ ]:
# 1. LinearRegression
lr = LinearRegression(featuresCol="scaled_features", labelCol="target_label", regParam=0.0)

# 2. GeneralizedLinearRegression
glr = GeneralizedLinearRegression(featuresCol="scaled_features", labelCol="target_label", family="gaussian", link="identity")

# 3. DecisionTreeRegressor
dt_reg = DecisionTreeRegressor(featuresCol="scaled_features", labelCol="target_label", maxDepth=5)

# 4. RandomForestRegressor
rf_reg = RandomForestRegressor(featuresCol="scaled_features", labelCol="target_label", numTrees=20)

# 5. GBTRegressor (Gradient-Boosted Trees)
gbt_reg = GBTRegressor(featuresCol="scaled_features", labelCol="target_label", maxIter=20)

### C. Supervised Learning (Classification Estimators)

In [ ]:
# 1. LogisticRegression (Binary and Multiclass)
log_reg = LogisticRegression(featuresCol="scaled_features", labelCol="binary_label", maxIter=100)

# 2. DecisionTreeClassifier
dt_class = DecisionTreeClassifier(featuresCol="scaled_features", labelCol="binary_label", maxDepth=5)

# 3. RandomForestClassifier
rf_class = RandomForestClassifier(featuresCol="scaled_features", labelCol="binary_label", numTrees=20)

# 4. GBTClassifier
gbt_class = GBTClassifier(featuresCol="scaled_features", labelCol="binary_label", maxIter=20)

# 5. LinearSupportVectorMachine (Binary classification)
lsvc = LinearSVC(featuresCol="scaled_features", labelCol="binary_label", maxIter=100)

# 6. NaiveBayes (Naive Bayes Classifier)
# Note: Naive Bayes requires non-negative feature features. We use scaled_features_minmax.
nb = NaiveBayes(featuresCol="scaled_features_minmax", labelCol="binary_label", modelType="multinomial")

### D. Unsupervised Learning (Clustering & Dimensionality Reduction)

In [ ]:
# 1. KMeans
kmeans = KMeans(featuresCol="scaled_features", k=2, seed=42)

# 2. BisectingKMeans (Hierarchical clustering)
bkm = BisectingKMeans(featuresCol="scaled_features", k=2, seed=42)

# 3. GaussianMixture (EM probability clustering)
gmm = GaussianMixture(featuresCol="scaled_features", k=2, seed=42)

# 4. LDA (Latent Dirichlet Allocation - Topic modeling)
lda = LDA(featuresCol="scaled_features", k=2, maxIter=10)

# 5. PCA (Principal Component Analysis)
pca = PCA(inputCol="scaled_features", outputCol="pca_features", k=2)

### E. Neural Networks (Deep Learning Estimator)

In [ ]:
# MultilayerPerceptronClassifier (MLP Neural Network Classifier)
# Architecture: [input_features, hidden_1, hidden_2, output_classes]
layers = [3, 4, 4, 2] # 3 features in input vector, hidden layers, 2 output labels (binary classification)

mlp = MultilayerPerceptronClassifier(
    featuresCol="scaled_features",
    labelCol="binary_label",
    layers=layers,
    blockSize=128,
    seed=42,
    maxIter=100,
    solver="gd"
)

### F. Pipeline Assembly, Splitting, Training & Prediction

In [ ]:
# Combine preprocessing stages and model stage into a Pipeline
pipeline = Pipeline(stages=[assembler, scaler, string_indexer, ohe, min_max_scaler, pca, lr])

# Split dataset into train and test splits
(trainingData, testingData) = df_ml.randomSplit([0.7, 0.3], seed=42)

# Fit Pipeline Model (Trains the estimator and parameters of transformers)
pipelineModel = pipeline.fit(trainingData)

# Transform / Predict on unseen testingData
predictions = pipelineModel.transform(testingData)
print("Pipeline Predictions completed. Outputs schema:")
predictions.printSchema()

### G. Evaluation Metrics

In [ ]:
# 1. Regression Evaluation
reg_evaluator = RegressionEvaluator(predictionCol="prediction", labelCol="target_label", metricName="rmse")
rmse_score = reg_evaluator.evaluate(predictions)
print(f"Regression RMSE Score: {rmse_score:.4f}")

# 2. Binary Classification Evaluation
# Fit a classification model first to obtain raw predictions and probabilities
class_pipeline = Pipeline(stages=[assembler, scaler, log_reg])
class_model = class_pipeline.fit(trainingData)
class_predictions = class_model.transform(testingData)

bin_evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="binary_label", metricName="areaUnderROC")
auc_roc = bin_evaluator.evaluate(class_predictions)
print(f"Binary Classification AUC ROC: {auc_roc:.4f}")

# 3. Multiclass Classification Evaluation
multi_evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="binary_label", metricName="accuracy")
accuracy_val = multi_evaluator.evaluate(class_predictions)
print(f"Multiclass Classification Accuracy: {accuracy_val:.4f}")

# 4. Clustering Evaluation
# Fit a clustering model to obtain prediction cluster assignments
cluster_pipeline = Pipeline(stages=[assembler, scaler, kmeans])
cluster_model = cluster_pipeline.fit(trainingData)
cluster_predictions = cluster_model.transform(testingData)

clust_evaluator = ClusteringEvaluator(featuresCol="scaled_features", predictionCol="prediction", metricName="silhouette")
silhouette_score = clust_evaluator.evaluate(cluster_predictions)
print(f"Clustering Silhouette Score: {silhouette_score:.4f}")

### H. Pipeline Model Persistence (Saving & Loading)

In [ ]:
# Save pipeline model to disk
pipelineModel.write().overwrite().save("generic_pipeline_model")

# Load model back
loadedPipelineModel = PipelineModel.load("generic_pipeline_model")
print("Model saved and loaded back successfully.")

## 8. Cleanup & Teardown

In [ ]:
# Stop Spark Session
spark.stop()
print("Spark session stopped.")

# Close ClearML Task
task.close()
print("ClearML task closed.")